# Trabalho 1 — Raízes de Equações
## Parte I: Métodos Intervalares

Referência: CHAPRA, S. C.; CANALE, R. P. *Métodos Numéricos para Engenharia*. McGraw-Hill, 2008.

| Seção | Item do enunciado |
|---|---|
| 0. Configuração | leitura da função digitada |
| 1. Varredura | item 1 |
| 2. Bisseção | item 2 |
| 3. Falsa posição | item 3 |
| 4. Comparação gráfica | item 4 |
| 5. Falsa posição modificada | item 5 |
| 6. Integração dos métodos | item 6 (a implementar) |
| 7. Testes de consistência | item 7 |
| 8. Programa interativo | itens 1 a 5 com entrada do usuário |

**Como usar:** execute as células em ordem (*Ambiente de execução → Executar tudo*). Cada seção define suas funções e, em seguida, mostra um exemplo. A seção 8 é o programa interativo, que pede os dados ao usuário.

Todo o cálculo é feito em **Python puro** (módulo `math`). Bibliotecas externas: `sympy`, só para interpretar o texto digitado, e `matplotlib`, para os gráficos. As duas já vêm instaladas no Colab.

# 0. Configuração

## Importações

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import (
    convert_xor,
    implicit_multiplication_application,
    parse_expr,
    standard_transformations,
)

## Leitura da função (cálculo simbólico)

O usuário digita a função como texto, por exemplo `x^2 - 2`. O **SymPy** transforma esse texto em uma expressão matemática (cálculo simbólico) e depois em uma função Python comum, que usa o módulo `math`. Nenhuma raiz é calculada aqui.

- aceita `x^2` (potência), `2x` (multiplicação implícita), `sen`, `tg`, `ln`, `e`, `pi`;
- pontos fora do domínio (`log(-1)`, `1/0`, `sqrt(-1)`) devolvem `nan` em vez de interromper o programa.

In [ ]:
X = sp.Symbol("x")

# Permite escrever "x^2" (em vez de x**2) e "2x" (em vez de 2*x).
_TRANSFORMACOES = standard_transformations + (
    convert_xor,
    implicit_multiplication_application,
)

# Nomes aceitos na expressão além de x.
_NOMES = {
    "x": X,
    "e": sp.E,
    "pi": sp.pi,
    "ln": sp.log,
    "log": sp.log,
    "log10": lambda arg: sp.log(arg, 10),
    "sen": sp.sin,
    "tg": sp.tan,
}


def ler_funcao(texto):
    """Converte o texto em (expressão simbólica, função f(x)).

    f recebe um número (float) e devolve um float. Fora do domínio
    (ex.: log(-1), 1/0, sqrt(-1)) devolve float("nan") em vez de dar erro.
    Lança ValueError se a expressão for inválida ou depender de outra variável.
    """
    try:
        expr = parse_expr(texto, local_dict=_NOMES, transformations=_TRANSFORMACOES)
    except Exception as erro:
        raise ValueError(f"expressão inválida: {erro}") from erro

    if not isinstance(expr, sp.Expr):
        raise ValueError("a entrada não é uma expressão matemática")
    extras = expr.free_symbols - {X}
    if extras:
        nomes = ", ".join(sorted(str(s) for s in extras))
        raise ValueError(f"a expressão deve depender apenas de x (encontrado: {nomes})")

    f_math = sp.lambdify(X, expr, modules="math")

    def f(x):
        try:
            valor = f_math(x)
        except (ValueError, ZeroDivisionError, OverflowError):
            return float("nan")
        # Ex.: (-1)**0.5 em Python dá um número complexo: fora do domínio real.
        if isinstance(valor, complex):
            return float("nan")
        return float(valor)

    return expr, f

In [ ]:
expr, f = ler_funcao("x^2 - 2")
print("Expressão:", expr)
print("f(3)  =", f(3))
print("log(-1) ->", ler_funcao("log(x)")[1](-1))

# 1. Varredura (busca incremental)

Seção 5.4 do livro. O intervalo [a, b] é dividido em **N** subintervalos e o **sinal** de f é calculado em cada ponto da malha. Um subintervalo é marcado quando:
- f **troca de sinal** entre suas extremidades (há raiz dentro dele, se f for contínua); ou
- f é **exatamente zero** em um ponto da malha.

| Função | O que faz |
|---|---|
| `gerar_malha` | pontos x₀ = a, …, x_N = b |
| `sinal` | −1, 0, +1, ou `None` se o valor não for finito |
| `calcular_sinais` | rotina do item 1.3: sinal de f em cada subdivisão |
| `varredura` | devolve os subintervalos com troca de sinal ou zero |
| `tabela_varredura` | tabela dos subintervalos detectados |
| `plotar_varredura` | gráfico com as subdivisões, destacando as que têm raiz |

In [ ]:
def gerar_malha(a, b, n):
    """Devolve os n + 1 pontos igualmente espaçados x0 = a, ..., xn = b."""
    h = (b - a) / n
    return [a + i * h for i in range(n)] + [b]


def sinal(valor):
    """Sinal de um número: -1, 0 ou +1; None se o valor não for finito."""
    if not math.isfinite(valor):
        return None
    if valor > 0:
        return 1
    if valor < 0:
        return -1
    return 0


def calcular_sinais(f, a, b, n):
    """Divide [a, b] em n subintervalos e devolve (pontos, valores, sinais).

    sinais[i] vale -1, 0 ou +1, ou None quando f(x_i) não é finito
    (por exemplo, fora do domínio ou em uma assíntota).
    """
    pontos = gerar_malha(a, b, n)
    valores = [f(x) for x in pontos]
    sinais = [sinal(v) for v in valores]
    return pontos, valores, sinais


def varredura(f, a, b, n):
    """Busca incremental de raízes em [a, b] com n subdivisões.

    Devolve uma lista de dicionários, um por subintervalo detectado:
        {"xl", "xu", "f_xl", "f_xu", "tipo"}
    em que "tipo" é "mudança de sinal" ou "zero exato" (quando f(xl) == 0 ou
    f(xu) == 0). Cada zero exato é reportado uma única vez.
    """
    if not a < b:
        raise ValueError("o intervalo deve satisfazer a < b")
    if n < 1:
        raise ValueError("N deve ser um inteiro positivo")

    pontos, valores, sinais = calcular_sinais(f, a, b, n)
    detectados = []
    for i in range(n):
        s_esq, s_dir = sinais[i], sinais[i + 1]
        if s_esq is None or s_dir is None:
            continue

        # Um zero no ponto x_i já foi reportado pelo subintervalo anterior,
        # exceto quando x_i é o extremo esquerdo do domínio.
        zero_esq_novo = s_esq == 0 and i == 0
        if s_esq * s_dir < 0:
            tipo = "mudança de sinal"
        elif s_dir == 0 or zero_esq_novo:
            tipo = "zero exato"
        else:
            continue

        detectados.append({
            "xl": pontos[i],
            "xu": pontos[i + 1],
            "f_xl": valores[i],
            "f_xu": valores[i + 1],
            "tipo": tipo,
        })
    return detectados


def tabela_varredura(detectados):
    """Tabela dos subintervalos detectados: xl, xu, f(xl), f(xu) e tipo."""
    cabecalho = (f"{'#':>3} | {'xl':>12} | {'xu':>12} | {'f(xl)':>12}"
                 f" | {'f(xu)':>12} | Tipo")
    linhas = [cabecalho, "-" * (len(cabecalho) + 14)]
    for k, s in enumerate(detectados, start=1):
        linhas.append(f"{k:>3} | {s['xl']:>12.6g} | {s['xu']:>12.6g}"
                      f" | {s['f_xl']:>12.4e} | {s['f_xu']:>12.4e} | {s['tipo']}")
    return "\n".join(linhas)


def percentil(valores, p):
    """Percentil p (0 a 100) de uma lista, pelo método do vizinho mais próximo."""
    ordenados = sorted(valores)
    indice = round(p / 100 * (len(ordenados) - 1))
    return ordenados[indice]


def plotar_varredura(f, expr, a, b, n, detectados, arquivo=None):
    """Plota f em [a, b], as n subdivisões e destaca os subintervalos detectados.

    Se `arquivo` for informado, a figura também é salva nesse caminho.
    """
    pontos, valores, sinais = calcular_sinais(f, a, b, n)

    # Curva suave: malha fina; valores não finitos viram NaN (o matplotlib
    # interrompe a linha nesses pontos).
    x_fino = gerar_malha(a, b, max(2000, 20 * n))
    y_fino = [f(x) for x in x_fino]
    y_fino = [y if math.isfinite(y) else float("nan") for y in y_fino]

    fig, ax = plt.subplots(figsize=(11, 5.5))

    for k, sub in enumerate(detectados):
        ax.axvspan(sub["xl"], sub["xu"], color="tab:orange", alpha=0.3,
                   label="subintervalo com raiz" if k == 0 else None)

    # Linhas verticais finas marcam as subdivisões (omitidas se forem muitas).
    if n <= 200:
        for xi in pontos:
            ax.axvline(xi, color="gray", lw=0.5, alpha=0.5)

    ax.axhline(0, color="black", lw=0.8)
    ax.plot(x_fino, y_fino, color="tab:blue", lw=1.6, label=f"f(x) = {expr}")

    grupos = [(1, "o", "tab:green", 4, "f(xᵢ) > 0"),
              (-1, "o", "tab:red", 4, "f(xᵢ) < 0"),
              (0, "D", "black", 6, "f(xᵢ) = 0")]
    for s_grupo, marcador, cor, tamanho, rotulo in grupos:
        xs = [x for x, s in zip(pontos, sinais) if s == s_grupo]
        ys = [v for v, s in zip(valores, sinais) if s == s_grupo]
        if xs:
            ax.plot(xs, ys, marcador, color=cor, ms=tamanho, label=rotulo)

    # Limita o eixo y para que assíntotas não esmaguem o gráfico.
    finitos = [abs(v) for v in valores if math.isfinite(v)]
    if finitos:
        limite = percentil(finitos, 95) * 1.5 or 1.0
        ax.set_ylim(-limite, limite)

    ax.set_xlim(a, b)
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.set_title(f"Varredura incremental — N = {n} subdivisões, "
                 f"{len(detectados)} subintervalo(s) detectado(s)")
    ax.legend(loc="best", fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()

    if arquivo:
        Path(arquivo).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(arquivo, dpi=150)
    return fig

## Exemplo — f(x) = sen(10x) + cos(3x) em [0, 5]

Com **N = 25** algumas raízes muito próximas passam despercebidas. Refinando para **N = 100**, elas aparecem (item 1.4; Figura 5.16 do livro).

In [ ]:
expr, f = ler_funcao("sin(10x) + cos(3x)")

for n in (25, 100):
    detectados = varredura(f, 0, 5, n)
    print(f"N = {n}: {len(detectados)} subintervalo(s) detectado(s)")
    print(tabela_varredura(detectados))
    plotar_varredura(f, expr, 0, 5, n, detectados)
    plt.show()

# 2. Método da Bisseção

Pseudocódigo da Figura 5.10 do livro. A cada iteração:

1. ponto médio: **xr = (xl + xu) / 2**;
2. se f(xl)·f(xr) < 0, a raiz está em [xl, xr], então `xu = xr`; senão, `xl = xr`;
3. erro relativo aproximado: **εa = |(xr_novo − xr_anterior) / xr_novo| × 100 %**;
4. para quando **εa < εs** (tolerância), f(xr) = 0 ou o limite de iterações é atingido.

Colunas da tabela: iteração, extremos **xl** e **xu**, ponto médio **xr**, **f(xr)** e **εa**.

In [ ]:
def bissecao(f, xl, xu, es, max_iter=100):
    """Calcula uma raiz de f em [xl, xu] pelo método da bisseção.

    Parâmetros
        f        : função de uma variável
        xl, xu   : extremos do intervalo, com f(xl) * f(xu) <= 0
        es       : tolerância (critério de parada) em porcentagem, εs > 0
        max_iter : limite de iterações

    Devolve um dicionário com
        "raiz"       : última aproximação xr
        "iteracoes"  : lista de linhas {iter, xl, xu, xr, f_xr, ea}
        "convergiu"  : True se εa < εs ou f(xr) == 0
        "motivo"     : texto explicando o critério de parada
    """
    if es <= 0:
        raise ValueError("a tolerância deve ser positiva")
    fl, fu = f(xl), f(xu)
    if fl * fu > 0:
        raise ValueError("f(xl) e f(xu) têm o mesmo sinal; não há raiz garantida")

    # Raiz exatamente em um dos extremos: nada a iterar.
    if fl == 0 or fu == 0:
        return {"raiz": xl if fl == 0 else xu, "iteracoes": [], "convergiu": True,
                "motivo": "f é exatamente zero em um extremo do intervalo"}

    iteracoes = []
    xr = None
    for it in range(1, max_iter + 1):
        xr_anterior = xr
        xr = (xl + xu) / 2
        fr = f(xr)

        # εa não é definido na primeira iteração (não há valor anterior).
        ea = None
        if xr_anterior is not None and xr != 0:
            ea = abs((xr - xr_anterior) / xr) * 100

        iteracoes.append({"iter": it, "xl": xl, "xu": xu, "xr": xr,
                          "f_xr": fr, "ea": ea})

        if fr == 0:
            return {"raiz": xr, "iteracoes": iteracoes, "convergiu": True,
                    "motivo": "f(xr) = 0 (raiz exata)"}
        if ea is not None and ea < es:
            return {"raiz": xr, "iteracoes": iteracoes, "convergiu": True,
                    "motivo": f"εa = {ea:.3g}% < εs = {es:g}%"}

        if fl * fr < 0:
            xu = xr
        else:
            xl, fl = xr, fr

    return {"raiz": xr, "iteracoes": iteracoes, "convergiu": False,
            "motivo": f"limite de {max_iter} iterações atingido"}


def tabela_bissecao(iteracoes):
    """Tabela do item 2.3: iteração, xl, xu, ponto médio, f(xr) e εa."""
    cabecalho = (f"{'Iter':>4} | {'xl':>16} | {'xu':>16} | {'xr (ponto médio)':>16}"
                 f" | {'f(xr)':>13} | {'εa (%)':>12}")
    linhas = [cabecalho, "-" * len(cabecalho)]
    for r in iteracoes:
        ea = "—" if r["ea"] is None else f"{r['ea']:.6f}"
        linhas.append(f"{r['iter']:>4} | {r['xl']:>16.10g} | {r['xu']:>16.10g}"
                      f" | {r['xr']:>16.10g} | {r['f_xr']:>13.6e} | {ea:>12}")
    return "\n".join(linhas)

## Exemplo — f(x) = x² − 2 em [1, 2], εs = 0,5 %

A raiz exata é √2 ≈ 1,41421.

In [ ]:
expr, f = ler_funcao("x^2 - 2")
resultado = bissecao(f, 1, 2, es=0.5)

print(tabela_bissecao(resultado["iteracoes"]))
print(f"\nRaiz ≈ {resultado['raiz']:.10g}  ({resultado['motivo']})")

## Exemplo — bisseção em todas as raízes detectadas na varredura

Item 2.1: os subintervalos da varredura são a entrada da bisseção.

In [ ]:
expr, f = ler_funcao("x^2 - 2")
detectados = varredura(f, -3, 3, 6)
print(tabela_varredura(detectados))

for k, sub in enumerate(detectados, start=1):
    resultado = bissecao(f, sub["xl"], sub["xu"], es=0.5)
    print(f"\n=== Raiz {k}: intervalo [{sub['xl']:g}, {sub['xu']:g}] ===")
    print(tabela_bissecao(resultado["iteracoes"]))
    print(f"Raiz ≈ {resultado['raiz']:.10g}")

# 3. Método da Falsa Posição

Seção 5.3 do livro. Em vez do ponto médio, usa a interseção com o eixo x da reta que liga (xl, f(xl)) e (xu, f(xu)) (Eq. 5.7):

$$x_r = x_u - \frac{f(x_u)\,(x_l - x_u)}{f(x_l) - f(x_u)}$$

O resto (atualização do intervalo, εa, critério de parada) é igual ao da bisseção.

In [ ]:
def falsa_posicao(f, xl, xu, es, max_iter=100):
    """Calcula uma raiz de f em [xl, xu] pelo método da falsa posição.

    Parâmetros
        f        : função de uma variável
        xl, xu   : extremos do intervalo, com f(xl) * f(xu) <= 0
        es       : tolerância (critério de parada) em porcentagem, εs > 0
        max_iter : limite de iterações

    Devolve um dicionário com
        "raiz"       : última aproximação xr
        "iteracoes"  : lista de linhas {iter, xl, xu, xr, f_xr, ea}
        "convergiu"  : True se εa < εs ou f(xr) == 0
        "motivo"     : texto explicando o critério de parada
    """
    if es <= 0:
        raise ValueError("a tolerância deve ser positiva")
    fl, fu = f(xl), f(xu)
    if fl * fu > 0:
        raise ValueError("f(xl) e f(xu) têm o mesmo sinal; não há raiz garantida")

    # Raiz exatamente em um dos extremos: nada a iterar.
    if fl == 0 or fu == 0:
        return {"raiz": xl if fl == 0 else xu, "iteracoes": [], "convergiu": True,
                "motivo": "f é exatamente zero em um extremo do intervalo"}

    iteracoes = []
    xr = None
    for it in range(1, max_iter + 1):
        xr_anterior = xr
        xr = xu - fu * (xl - xu) / (fl - fu)
        fr = f(xr)

        # εa não é definido na primeira iteração (não há valor anterior).
        ea = None
        if xr_anterior is not None and xr != 0:
            ea = abs((xr - xr_anterior) / xr) * 100

        iteracoes.append({"iter": it, "xl": xl, "xu": xu, "xr": xr,
                          "f_xr": fr, "ea": ea})

        if fr == 0:
            return {"raiz": xr, "iteracoes": iteracoes, "convergiu": True,
                    "motivo": "f(xr) = 0 (raiz exata)"}
        if ea is not None and ea < es:
            return {"raiz": xr, "iteracoes": iteracoes, "convergiu": True,
                    "motivo": f"εa = {ea:.3g}% < εs = {es:g}%"}

        if fl * fr < 0:
            xu, fu = xr, fr
        else:
            xl, fl = xr, fr

    return {"raiz": xr, "iteracoes": iteracoes, "convergiu": False,
            "motivo": f"limite de {max_iter} iterações atingido"}


def tabela_falsa_posicao(iteracoes):
    """Tabela de iterações: iteração, xl, xu, xr, f(xr) e εa."""
    cabecalho = (f"{'Iter':>4} | {'xl':>16} | {'xu':>16} | {'xr':>16}"
                 f" | {'f(xr)':>13} | {'εa (%)':>12}")
    linhas = [cabecalho, "-" * len(cabecalho)]
    for r in iteracoes:
        ea = "—" if r["ea"] is None else f"{r['ea']:.6f}"
        linhas.append(f"{r['iter']:>4} | {r['xl']:>16.10g} | {r['xu']:>16.10g}"
                      f" | {r['xr']:>16.10g} | {r['f_xr']:>13.6e} | {ea:>12}")
    return "\n".join(linhas)

## Exemplo — mesma função da bisseção: f(x) = x² − 2 em [1, 2], εs = 0,5 %

A falsa posição precisa de 4 iterações; a bisseção precisou de 8.

In [ ]:
expr, f = ler_funcao("x^2 - 2")
resultado = falsa_posicao(f, 1, 2, es=0.5)

print(tabela_falsa_posicao(resultado["iteracoes"]))
print(f"\nRaiz ≈ {resultado['raiz']:.10g}  ({resultado['motivo']})")

# 4. Comparação gráfica: bisseção × falsa posição

Semelhante à Figura 5.13 do livro: o **erro relativo percentual verdadeiro εt** (escala log) por iteração. Como εt exige a raiz exata, `raiz_referencia` a calcula com precisão de máquina.

| Função | O que faz |
|---|---|
| `raiz_referencia` | raiz "exata" (bisseção até o limite do ponto flutuante) |
| `erro_verdadeiro` | εt = \|(raiz − xr) / raiz\| × 100 % |
| `comparar` | roda vários métodos no mesmo intervalo |
| `tabela_resumo` | iterações, raiz, f(raiz), εa e εt finais de cada método |
| `plotar_comparacao` | gráficos de εt e εa por iteração |

In [ ]:
# Métodos pedidos no item 4. Outros dicionários podem ser passados a `comparar`.
METODOS_ITEM4 = {"Bisseção": bissecao, "Falsa posição": falsa_posicao}


def raiz_referencia(f, xl, xu):
    """Estima a raiz em [xl, xu] com precisão de máquina (bisseção sem tolerância).

    Para quando o ponto médio não difere mais dos extremos em ponto flutuante.
    """
    fl, fu = f(xl), f(xu)
    if fl == 0:
        return xl
    if fu == 0:
        return xu
    for _ in range(2000):
        xm = (xl + xu) / 2
        if xm in (xl, xu):
            break
        fm = f(xm)
        if fm == 0:
            return xm
        if fl * fm < 0:
            xu = xm
        else:
            xl, fl = xm, fm
    return (xl + xu) / 2


def erro_verdadeiro(xr, raiz):
    """εt = |(raiz - xr) / raiz| * 100 %; erro absoluto se a raiz for 0."""
    if raiz == 0:
        return abs(xr)
    return abs((raiz - xr) / raiz) * 100


def comparar(f, xl, xu, es, max_iter=100, metodos=METODOS_ITEM4):
    """Executa cada método no mesmo intervalo.

    Devolve (raiz_ref, {nome: resultado}).
    """
    raiz = raiz_referencia(f, xl, xu)
    resultados = {nome: metodo(f, xl, xu, es, max_iter)
                  for nome, metodo in metodos.items()}
    return raiz, resultados


def tabela_resumo(f, raiz, resultados):
    """Tabela com iterações, raiz obtida, f(raiz), εa e εt finais de cada método.

    Comparar εa com εt mostra se o critério de parada foi confiável: se
    εt > εa, o método parou achando estar mais perto da raiz do que estava.
    """
    cabecalho = (f"{'Método':<26} | {'Iterações':>9} | {'Raiz':>16} | {'f(raiz)':>10}"
                 f" | {'εa final (%)':>12} | {'εt final (%)':>12} | Convergiu")
    linhas = [cabecalho, "-" * (len(cabecalho) + 2)]
    for nome, res in resultados.items():
        ea = res["iteracoes"][-1]["ea"] if res["iteracoes"] else None
        ea_txt = "—" if ea is None else f"{ea:.3e}"
        et = erro_verdadeiro(res["raiz"], raiz)
        linhas.append(f"{nome:<26} | {len(res['iteracoes']):>9} | {res['raiz']:>16.10g}"
                      f" | {f(res['raiz']):>10.3e} | {ea_txt:>12} | {et:>12.3e}"
                      f" | {'sim' if res['convergiu'] else 'não'}")
    return "\n".join(linhas)


def plotar_comparacao(expr, xl, xu, es, raiz, resultados, arquivo=None, titulo=None):
    """Gráfico de εt e εa por iteração para cada método (escala log em y).

    Iterações com erro exatamente zero não são desenhadas (log de 0).
    """
    fig, (ax_t, ax_a) = plt.subplots(1, 2, figsize=(12, 5))
    marcadores = ["o", "s", "^", "D"]

    for k, (nome, res) in enumerate(resultados.items()):
        its = [r["iter"] for r in res["iteracoes"]]
        et = [erro_verdadeiro(r["xr"], raiz) for r in res["iteracoes"]]
        ea = [r["ea"] if r["ea"] is not None else 0 for r in res["iteracoes"]]
        et = [e if e > 0 else float("nan") for e in et]
        ea = [e if e > 0 else float("nan") for e in ea]
        rotulo = f"{nome} ({len(its)} iter.)"
        m = marcadores[k % len(marcadores)]
        ax_t.semilogy(its, et, marker=m, ms=4, label=rotulo)
        ax_a.semilogy(its, ea, marker=m, ms=4, label=rotulo)

    ax_a.axhline(es, color="black", ls="--", lw=1, label=f"εs = {es:g}%")

    ax_t.set_title("Erro relativo percentual verdadeiro εt")
    ax_a.set_title("Erro relativo percentual aproximado εa")
    for ax in (ax_t, ax_a):
        ax.set_xlabel("Iterações")
        ax.set_ylabel("Erro (%)")
        ax.grid(alpha=0.3, which="both")
        ax.legend(fontsize=9)
        ax.xaxis.get_major_locator().set_params(integer=True)

    fig.suptitle(titulo or f"f(x) = {expr} em [{xl:g}, {xu:g}] — raiz ≈ {raiz:.10g}")
    fig.tight_layout()

    if arquivo:
        Path(arquivo).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(arquivo, dpi=150)
    return fig

## Exemplo — f(x) = e⁻ˣ − x em [0, 1], εs = 0,01 %

Caso típico: a falsa posição converge bem mais rápido.

In [ ]:
expr, f = ler_funcao("exp(-x) - x")
raiz, resultados = comparar(f, 0, 1, es=0.01)

print(f"Raiz de referência: {raiz:.15g}\n")
print(tabela_resumo(f, raiz, resultados))
plotar_comparacao(expr, 0, 1, 0.01, raiz, resultados)
plt.show()

# 5. Método da Falsa Posição Modificado

Seção 5.3.2 e Figura 5.15 do livro. A falsa posição pode ficar **unilateral**: um extremo fica parado e a convergência fica lenta. Na versão modificada, se o mesmo extremo ficar parado por **duas iterações seguidas**, o valor de f nesse extremo é **dividido por 2**, o que inclina a reta e destrava o método.

A coluna **Obs.** da tabela mostra em quais iterações a divisão aconteceu.

In [ ]:
def falsa_posicao_modificada(f, xl, xu, es, max_iter=100):
    """Calcula uma raiz de f em [xl, xu] pela falsa posição modificada.

    Parâmetros
        f        : função de uma variável
        xl, xu   : extremos do intervalo, com f(xl) * f(xu) <= 0
        es       : tolerância (critério de parada) em porcentagem, εs > 0
        max_iter : limite de iterações

    Devolve um dicionário com
        "raiz"       : última aproximação xr
        "iteracoes"  : lista de linhas {iter, xl, xu, xr, f_xr, ea, obs}
                       ("obs" indica quando f(xl) ou f(xu) foi dividido por 2)
        "convergiu"  : True se εa < εs ou f(xr) == 0
        "motivo"     : texto explicando o critério de parada
    """
    if es <= 0:
        raise ValueError("a tolerância deve ser positiva")
    fl, fu = f(xl), f(xu)
    if fl * fu > 0:
        raise ValueError("f(xl) e f(xu) têm o mesmo sinal; não há raiz garantida")

    # Raiz exatamente em um dos extremos: nada a iterar.
    if fl == 0 or fu == 0:
        return {"raiz": xl if fl == 0 else xu, "iteracoes": [], "convergiu": True,
                "motivo": "f é exatamente zero em um extremo do intervalo"}

    iteracoes = []
    xr = None
    il = iu = 0  # quantas iterações seguidas xl / xu ficaram parados
    for it in range(1, max_iter + 1):
        xr_anterior = xr
        # fl e fu são os valores guardados (podem já ter sido divididos por 2).
        xr = xu - fu * (xl - xu) / (fl - fu)
        fr = f(xr)

        # εa não é definido na primeira iteração (não há valor anterior).
        ea = None
        if xr_anterior is not None and xr != 0:
            ea = abs((xr - xr_anterior) / xr) * 100

        linha = {"iter": it, "xl": xl, "xu": xu, "xr": xr, "f_xr": fr,
                 "ea": ea, "obs": ""}
        iteracoes.append(linha)

        if fr == 0:
            return {"raiz": xr, "iteracoes": iteracoes, "convergiu": True,
                    "motivo": "f(xr) = 0 (raiz exata)"}
        if ea is not None and ea < es:
            return {"raiz": xr, "iteracoes": iteracoes, "convergiu": True,
                    "motivo": f"εa = {ea:.3g}% < εs = {es:g}%"}

        if fl * fr < 0:
            # Raiz em [xl, xr]: xu se move, xl fica parado.
            xu, fu = xr, fr
            iu, il = 0, il + 1
            if il >= 2:
                fl /= 2
                linha["obs"] = "xl parado: f(xl) ÷ 2"
        else:
            # Raiz em [xr, xu]: xl se move, xu fica parado.
            xl, fl = xr, fr
            il, iu = 0, iu + 1
            if iu >= 2:
                fu /= 2
                linha["obs"] = "xu parado: f(xu) ÷ 2"

    return {"raiz": xr, "iteracoes": iteracoes, "convergiu": False,
            "motivo": f"limite de {max_iter} iterações atingido"}


def tabela_falsa_posicao_modificada(iteracoes):
    """Tabela de iterações com a coluna Obs. (quando f foi dividido por 2)."""
    cabecalho = (f"{'Iter':>4} | {'xl':>16} | {'xu':>16} | {'xr':>16}"
                 f" | {'f(xr)':>13} | {'εa (%)':>12} | Obs.")
    linhas = [cabecalho, "-" * (len(cabecalho) + 18)]
    for r in iteracoes:
        ea = "—" if r["ea"] is None else f"{r['ea']:.6f}"
        linhas.append(f"{r['iter']:>4} | {r['xl']:>16.10g} | {r['xu']:>16.10g}"
                      f" | {r['xr']:>16.10g} | {r['f_xr']:>13.6e} | {ea:>12}"
                      f" | {r['obs']}")
    return "\n".join(linhas)

## Exemplo — f(x) = x¹⁰ − 1 em [0; 1,3], εs = 0,01 % (Exemplo 5.6 do livro)

Neste caso a falsa posição comum é lenta (39 iterações). A modificada resolve em 12, e a bisseção leva 14, os mesmos números do livro.

In [ ]:
expr, f = ler_funcao("x^10 - 1")

resultado = falsa_posicao_modificada(f, 0, 1.3, es=0.01)
print(tabela_falsa_posicao_modificada(resultado["iteracoes"]))
print(f"\nRaiz ≈ {resultado['raiz']:.10g}  ({resultado['motivo']})\n")

for nome, metodo in [("Bisseção", bissecao), ("Falsa posição", falsa_posicao),
                     ("Falsa posição modificada", falsa_posicao_modificada)]:
    print(f"{nome:<26}: {len(metodo(f, 0, 1.3, 0.01)['iteracoes'])} iterações")

# 6. Integração dos métodos *(a implementar)*

Rotina que recebe todos os subintervalos da varredura, calcula cada raiz com os três métodos, mostra uma tabela por raiz por método e indica qual método exigiu menos iterações.

Os três métodos têm a mesma assinatura `metodo(f, xl, xu, es, max_iter)` e devolvem o mesmo formato `{"raiz", "iteracoes", "convergiu", "motivo"}`, então a rotina pode chamá-los da mesma forma.

# 7. Testes de consistência

Exemplos com funções elementares que mostram vantagens e limitações de cada método:

| # | Função | O que mostra |
|---|---|---|
| 1 | x − 1 | falsa posição é exata para funções lineares |
| 2 | e⁻ˣ − x | caso típico: falsa posição mais rápida |
| 3 | cos x − x | com tolerância pequena a diferença aumenta |
| 4 | x¹⁰ − 1 | falsa posição unilateral (lenta); a modificada corrige |
| 5 | eˣ − 1000 | idem, com crescimento exponencial |
| 6 | tan x | polo: os métodos "convergem" para π/2 sem ser raiz |
| 7 | sen 10x + cos 3x | varredura com N pequeno perde raízes próximas |
| 8 | (x − 1)² | raiz dupla: não há troca de sinal |

In [ ]:
PASTA = "figuras/testes"

TODOS_OS_METODOS = {
    "Bisseção": bissecao,
    "Falsa posição": falsa_posicao,
    "Falsa posição modificada": falsa_posicao_modificada,
}

# Casos em que existe troca de sinal: os três métodos são executados e comparados.
CASOS_COMPARATIVOS = [
    {
        "titulo": "Função linear — a falsa posição é exata",
        "expr": "x - 1", "a": 0, "b": 3, "es": 0.01,
        "conclusao": "Para f linear, a reta da falsa posição coincide com a própria "
                     "função: a raiz sai na 1ª iteração. A bisseção não usa os "
                     "valores de f e precisa de ~15 iterações mesmo assim.",
    },
    {
        "titulo": "Caso típico — falsa posição mais rápida",
        "expr": "exp(-x) - x", "a": 0, "b": 1, "es": 0.01,
        "conclusao": "Função suave e com pouca curvatura: a falsa posição converge "
                     "em bem menos iterações que a bisseção (como na Figura 5.13).",
    },
    {
        "titulo": "Caso típico com tolerância muito pequena",
        "expr": "cos(x) - x", "a": 0, "b": 1, "es": 1e-6,
        "conclusao": "Com εs = 1e-6 %, a bisseção precisa de muitas iterações (o "
                     "intervalo só cai pela metade a cada passo), enquanto a falsa "
                     "posição usa a forma de f e chega lá muito antes.",
    },
    {
        "titulo": "Limitação da falsa posição — curvatura forte (Exemplo 5.6)",
        "expr": "x^10 - 1", "a": 0, "b": 1.3, "es": 0.01,
        "conclusao": "xu fica preso em 1,3 e xl se arrasta até a raiz: a falsa posição "
                     "é unilateral e fica mais lenta que a bisseção. Além disso, εa "
                     "final < εt final: o critério de parada engana. A versão "
                     "modificada divide f(xu) por 2 e resolve o problema.",
    },
    {
        "titulo": "Limitação da falsa posição — crescimento exponencial",
        "expr": "exp(x) - 1000", "a": 0, "b": 10, "es": 0.01,
        "conclusao": "f(xu) ≈ 21000 domina a reta, então xr avança muito pouco a cada "
                     "passo. A falsa posição é a mais lenta; a modificada é a mais rápida.",
    },
    {
        "titulo": "Limitação de todos — descontinuidade (polo)",
        "expr": "tan(x)", "a": 1, "b": 2, "es": 0.01,
        "conclusao": "tan(x) troca de sinal em π/2 sem ter raiz. Os três métodos "
                     "'convergem' para o polo e |f(raiz)| é enorme. Por isso é "
                     "sempre preciso conferir f(raiz) ≈ 0 (seção 5.3.1 do livro).",
    },
]

# Casos que afetam a etapa de varredura (antes dos métodos).
CASOS_VARREDURA = [
    {
        "titulo": "Limitação da varredura — raízes muito próximas",
        "expr": "sin(10x) + cos(3x)", "a": 3, "b": 5, "ns": [10, 100],
        "conclusao": "Com N pequeno, duas raízes no mesmo subintervalo se cancelam e "
                     "não há troca de sinal. Refinar N (item 1.4) revela as raízes "
                     "perdidas (Figura 5.16 do livro).",
    },
    {
        "titulo": "Limitação de todos — raiz dupla (sem troca de sinal)",
        "expr": "(x - 1)^2", "a": 0, "b": 3, "ns": [10, 12],
        "conclusao": "f(x) = (x−1)² toca o eixo sem cruzá-lo. A varredura só acha a "
                     "raiz se um ponto da malha cair exatamente em x = 1 (N = 12), e "
                     "nenhum método intervalar pode ser iniciado, pois f(xl)·f(xu) > 0.",
    },
]


def cabecalho(numero, titulo, expr):
    print("\n" + "=" * 78)
    print(f"Teste {numero}: {titulo}")
    print(f"f(x) = {expr}")
    print("=" * 78)


def executar_caso_comparativo(numero, caso):
    """Roda os três métodos no mesmo intervalo, imprime o resumo e salva o gráfico."""
    expr, f = ler_funcao(caso["expr"])
    a, b, es = caso["a"], caso["b"], caso["es"]
    cabecalho(numero, caso["titulo"], expr)
    print(f"Intervalo [{a}, {b}], εs = {es:g}%")

    raiz, resultados = comparar(f, a, b, es, max_iter=1000, metodos=TODOS_OS_METODOS)
    print(f"Raiz de referência: {raiz:.15g}\n")
    print(tabela_resumo(f, raiz, resultados))

    arquivo = f"{PASTA}/teste{numero}.png"
    fig = plotar_comparacao(expr, a, b, es, raiz, resultados, arquivo,
                            titulo=f"Teste {numero}: {caso['titulo']}")
    plt.show()
    print(f"\nConclusão: {caso['conclusao']}")
    print(f"Gráfico: {arquivo}")


def executar_caso_varredura(numero, caso):
    """Roda a varredura com vários N e testa a bisseção no intervalo inteiro."""
    expr, f = ler_funcao(caso["expr"])
    a, b = caso["a"], caso["b"]
    cabecalho(numero, caso["titulo"], expr)

    for n in caso["ns"]:
        detectados = varredura(f, a, b, n)
        intervalos = ", ".join(f"[{d['xl']:.4g}, {d['xu']:.4g}] ({d['tipo']})"
                               for d in detectados) or "nenhum"
        print(f"N = {n:>4}: {len(detectados)} subintervalo(s): {intervalos}")
        arquivo = f"{PASTA}/teste{numero}_N{n}.png"
        plotar_varredura(f, expr, a, b, n, detectados, arquivo)
        plt.show()
        print(f"          gráfico: {arquivo}")

    try:
        bissecao(f, a, b, 0.01)
        print(f"Bisseção em [{a}, {b}]: executada (há troca de sinal nos extremos).")
    except ValueError as erro:
        print(f"Bisseção em [{a}, {b}]: recusada — {erro}.")

    print(f"\nConclusão: {caso['conclusao']}")


def executar_testes():
    print("TESTES DE CONSISTÊNCIA — MÉTODOS INTERVALARES (item 7)")
    numero = 1
    for caso in CASOS_COMPARATIVOS:
        executar_caso_comparativo(numero, caso)
        numero += 1
    for caso in CASOS_VARREDURA:
        executar_caso_varredura(numero, caso)
        numero += 1

In [ ]:
executar_testes()

# 8. Programa interativo (itens 1 a 5)

Fluxo:
1. o usuário digita f(x), o intervalo [a, b] e N;
2. a varredura mostra a tabela e o gráfico, e pergunta se deseja **[r] refinar** com outro N ou **[c] prosseguir**;
3. menu: **[1]** bisseção, **[2]** falsa posição, **[3]** falsa posição modificada, **[4]** comparação gráfica, **[v]** nova varredura, **[s]** sair;
4. para cada raiz, mostra a tabela de iterações e pergunta se deseja continuar para a próxima.

Dicas de digitação: `x^2`, `2x`, `sen(x)`, `ln(x)`, `exp(x)`; decimais podem usar vírgula (`0,5`). Para interromper, use o botão ■ da célula.

In [ ]:
def mostrar_figura(fig):
    """No notebook, o gráfico é exibido logo abaixo da célula."""
    plt.show()


PASTA_FIGURAS = "figuras"
MAX_ITER_PADRAO = 100

# Opção do menu -> (nome, função do método, função que monta a tabela)
METODOS = {
    "1": ("Bisseção", bissecao, tabela_bissecao),
    "2": ("Falsa posição", falsa_posicao, tabela_falsa_posicao),
    "3": ("Falsa posição modificada", falsa_posicao_modificada,
          tabela_falsa_posicao_modificada),
}


def perguntar(mensagem, converter=str, validar=None, padrao=None):
    """Lê uma entrada do usuário até que ela seja válida.

    `converter` transforma o texto (ex.: float, int); `validar` devolve uma
    mensagem de erro ou None. Se `padrao` for dado, Enter vazio o aceita.
    """
    while True:
        sufixo = f" [{padrao}]" if padrao is not None else ""
        texto = input(f"{mensagem}{sufixo}: ").strip()
        if not texto and padrao is not None:
            texto = str(padrao)
        try:
            valor = converter(texto)
        except ValueError as erro:
            print(f"  Entrada inválida: {erro}")
            continue
        erro = validar(valor) if validar else None
        if erro:
            print(f"  {erro}")
            continue
        return valor


def numero(texto):
    """Converte texto em float aceitando vírgula decimal (ex.: 0,5)."""
    return float(texto.replace(",", "."))


def sim_ou_nao(mensagem, padrao="s"):
    resposta = perguntar(mensagem + " (s/n)", padrao=padrao,
                         validar=lambda r: None if r.lower() in ("s", "n")
                         else "Responda 's' ou 'n'.")
    return resposta.lower() == "s"


def etapa_varredura():
    """Item 1: lê f, [a, b] e N; repete a varredura até o usuário prosseguir."""
    expr, f = perguntar("Digite f(x) (ex.: x^3 - 2x - 5, sin(x) - x/2)", ler_funcao)
    print(f"  f(x) = {expr}")

    a = perguntar("Extremo inferior a", numero)
    b = perguntar("Extremo superior b", numero,
                  validar=lambda v: None if v > a else "b deve ser maior que a.")

    n = perguntar("Número de subdivisões N", int,
                  validar=lambda v: None if v >= 1 else "N deve ser >= 1.")
    while True:
        detectados = varredura(f, a, b, n)
        imprimir_varredura(detectados, n)

        arquivo = f"{PASTA_FIGURAS}/varredura_N{n}.png"
        fig = plotar_varredura(f, expr, a, b, n, detectados, arquivo)
        print(f"  Gráfico salvo em {arquivo}")
        mostrar_figura(fig)

        print("\nO que deseja fazer?")
        print("  [r] Refinar o rastreamento com outro valor de N")
        print("  [c] Prosseguir para o cálculo das raízes detectadas")
        opcao = perguntar("Opção", validar=lambda o: None if o.lower() in ("r", "c")
                          else "Escolha 'r' ou 'c'.").lower()
        if opcao == "c":
            return expr, f, detectados
        n = perguntar("Novo valor de N", int,
                      validar=lambda v: None if v >= 1 else "N deve ser >= 1.")


def imprimir_varredura(detectados, n):
    print(f"\n=== Varredura com N = {n} ===")
    if not detectados:
        print("Nenhuma mudança de sinal ou zero detectado. "
              "Tente aumentar N ou mudar o intervalo.")
        return
    print(tabela_varredura(detectados))


def ler_tolerancia():
    """Lê εs (%) e o limite de iterações, com valores padrão."""
    es = perguntar("Tolerância εs (%)", numero, padrao=0.5,
                   validar=lambda v: None if v > 0 else "A tolerância deve ser > 0.")
    max_iter = perguntar("Limite de iterações", int, padrao=MAX_ITER_PADRAO,
                         validar=lambda v: None if v >= 1 else "Deve ser >= 1.")
    return es, max_iter


def etapa_metodo(f, detectados, nome, metodo, tabela):
    """Itens 2, 3 e 5: aplica um método em cada subintervalo detectado."""
    print(f"\n##### {nome} #####")
    es, max_iter = ler_tolerancia()

    raizes = []
    for k, sub in enumerate(detectados, start=1):
        print(f"\n=== {nome} — raiz {k} de {len(detectados)} — intervalo "
              f"[{sub['xl']:.6g}, {sub['xu']:.6g}] ===")
        resultado = metodo(f, sub["xl"], sub["xu"], es, max_iter)
        if resultado["iteracoes"]:
            print(tabela(resultado["iteracoes"]))
        print(f"Raiz ≈ {resultado['raiz']:.10g}  "
              f"(f = {f(resultado['raiz']):.3e}; {resultado['motivo']})")
        if not resultado["convergiu"]:
            print("  Atenção: a tolerância não foi atingida.")
        # Numa assíntota (ex.: tan x em π/2) f troca de sinal sem ter raiz, e
        # |f(xr)| cresce em vez de diminuir.
        if abs(f(resultado["raiz"])) > max(abs(sub["f_xl"]), abs(sub["f_xu"])):
            print("  Atenção: |f(xr)| aumentou em relação aos extremos — "
                  "possível descontinuidade (polo), não uma raiz.")
        raizes.append(resultado["raiz"])

        if k < len(detectados) and not sim_ou_nao(
                "Deseja repetir o processo para a próxima raiz?"):
            break

    print(f"\n=== Resumo das raízes calculadas ({nome}) ===")
    for k, r in enumerate(raizes, start=1):
        print(f"  x{k} ≈ {r:.10g}")


def etapa_comparacao(f, expr, detectados):
    """Item 4: compara bisseção e falsa posição em um subintervalo escolhido."""
    print("\n##### Comparação gráfica: bisseção × falsa posição #####")
    if len(detectados) == 1:
        k = 1
    else:
        k = perguntar(f"Qual subintervalo comparar (1 a {len(detectados)})", int,
                      validar=lambda v: None if 1 <= v <= len(detectados)
                      else "Número fora da lista.")
    sub = detectados[k - 1]
    es, max_iter = ler_tolerancia()

    raiz, resultados = comparar(f, sub["xl"], sub["xu"], es, max_iter)
    print(f"\nRaiz de referência (precisão de máquina): {raiz:.15g}")
    print(tabela_resumo(f, raiz, resultados))

    arquivo = f"{PASTA_FIGURAS}/comparacao_raiz{k}.png"
    fig = plotar_comparacao(expr, sub["xl"], sub["xu"], es, raiz, resultados, arquivo)
    print(f"  Gráfico salvo em {arquivo}")
    mostrar_figura(fig)


def menu_metodos(f, expr, detectados):
    """Menu principal após a varredura. Devolve True para fazer nova varredura."""
    opcoes = {**{k: v[0] for k, v in METODOS.items()},
              "4": "Comparação gráfica bisseção × falsa posição",
              "v": "Nova varredura (outra função ou intervalo)",
              "s": "Sair"}
    while True:
        print("\nEscolha o método:")
        for chave, texto in opcoes.items():
            print(f"  [{chave}] {texto}")
        opcao = perguntar("Opção", validar=lambda o: None if o.lower() in opcoes
                          else "Opção inválida.").lower()
        if opcao == "s":
            return False
        if opcao == "v":
            return True
        if not detectados:
            print("Não há subintervalos detectados. Faça uma nova varredura.")
        elif opcao == "4":
            etapa_comparacao(f, expr, detectados)
        else:
            etapa_metodo(f, detectados, *METODOS[opcao])


def main():
    print("TRABALHO 1 — RAÍZES DE EQUAÇÕES")
    print("Parte I: métodos intervalares\n")
    while True:
        expr, f, detectados = etapa_varredura()
        if not menu_metodos(f, expr, detectados):
            break

In [ ]:
try:
    main()
except (KeyboardInterrupt, EOFError):
    print("\nEncerrado pelo usuário.")